In [1]:
import torch

C:\Users\David\AppData\Roaming\Python\Python312\site-packages\torch\_subclasses\functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
X = torch.ones([2,2], dtype=torch.float32)*2 # broadcasting in moltiplicazione
Y = torch.ones([2,2], dtype=torch.float32)*3 # broadcasting in moltiplicazione

In [4]:
X

tensor([[2., 2.],
        [2., 2.]])

In [5]:
Y

tensor([[3., 3.],
        [3., 3.]])

In [6]:
X*Y

tensor([[6., 6.],
        [6., 6.]])

In [7]:
X+Y

tensor([[5., 5.],
        [5., 5.]])

il prodotto tensoriale è diverso da quello matriciale

Il broadcasting invece è una tecnica che espande automaticamente il tensore per adattarlo a calcoli tra tensori

In [8]:
Xb = torch.ones([2,2], dtype=torch.float32)*2 # broadcasting in moltiplicazione
Yb = torch.ones([2,1], dtype=torch.float32)*3 # broadcasting in

In [9]:
Xb

tensor([[2., 2.],
        [2., 2.]])

In [10]:
Yb

tensor([[3.],
        [3.]])

in questo caso non sarebbero possibili operazioni tra tensori, però pytorch effettua il broadcasting, quindi riadatta il tensore Yb al tensore Xb

In [11]:
Xb*Yb

tensor([[6., 6.],
        [6., 6.]])

In [12]:
Xb+Yb

tensor([[5., 5.],
        [5., 5.]])

Il broadCasting funziona anche tra un vettore e un tensore

In [13]:
Xbv = torch.ones([2,2], dtype=torch.float32)*2 # broadcasting in moltiplicazione
Ybv = torch.ones([2], dtype=torch.float32)*3 # broadcasting in

In [14]:
Xbv+Ybv

tensor([[5., 5.],
        [5., 5.]])

In [15]:
Xbv*Ybv

tensor([[6., 6.],
        [6., 6.]])

ma non sempre il broadcasting funziona

In [16]:
Xbw = torch.ones([2,3,2,4], dtype=torch.float32)*2
Ybw = torch.ones([2], dtype=torch.float32)*3

In [17]:
try:
  Xbw+Ybw
except RuntimeError as e:
  print(f"RuntimeError: {e}")

RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 3


In [18]:
try:
  Xbw*Ybw
except RuntimeError as e:
  print(f"RuntimeError: {e}")

RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 3


In questo caso il broadcasting non è riuscito a riequilibrare gli shape dei tensori.

Adesso potremmo utilizzare la funzione unsqueeze per effettuare un reshape manuale

In [19]:
Xbw.shape

torch.Size([2, 3, 2, 4])

In [20]:
Ybw.shape

torch.Size([2])

In [21]:
print(torch.unsqueeze(Ybw, 0).shape)
print(torch.unsqueeze(Ybw, 1).shape)

torch.Size([1, 2])
torch.Size([2, 1])


In [22]:
Xbw+torch.unsqueeze(Ybw, 1)

tensor([[[[5., 5., 5., 5.],
          [5., 5., 5., 5.]],

         [[5., 5., 5., 5.],
          [5., 5., 5., 5.]],

         [[5., 5., 5., 5.],
          [5., 5., 5., 5.]]],


        [[[5., 5., 5., 5.],
          [5., 5., 5., 5.]],

         [[5., 5., 5., 5.],
          [5., 5., 5., 5.]],

         [[5., 5., 5., 5.],
          [5., 5., 5., 5.]]]])

Avendo aggiunto una dimensione siamo riusciti a far effettuare il broadcasting

Se volessi rimuovere dimensioni a 1 potrei usare lo squeeze

In [23]:
X_to_squeeze = torch.ones([2,3,1,2,1,4,1], dtype=torch.float32)*2

X_to_squeeze.shape

torch.Size([2, 3, 1, 2, 1, 4, 1])

In [24]:
torch.squeeze(X_to_squeeze).shape

torch.Size([2, 3, 2, 4])

Quando vediamo un tensore in realtà vediamo una vista della memoria.
Gli elementi di un tensore sono salvati in un array con celle di memoria contigue.
Per capire la colonna successiva si basa su un indice e uno stride che rappresenta il nr di elementi da saltare per passare al prossimo.
Il tensore contiene il puntatore all'inizio del blocco di dati

Lato python è possibile vedere una parte del tensore

In [25]:
X_elements = torch.ones([10,3], dtype=torch.float32)*torch.tensor([1,2,3], dtype=torch.float32)*torch.arange(1,11, dtype=torch.float32).view(10,1)

In [26]:
X_elements

tensor([[ 1.,  2.,  3.],
        [ 2.,  4.,  6.],
        [ 3.,  6.,  9.],
        [ 4.,  8., 12.],
        [ 5., 10., 15.],
        [ 6., 12., 18.],
        [ 7., 14., 21.],
        [ 8., 16., 24.],
        [ 9., 18., 27.],
        [10., 20., 30.]])

In [27]:
X_elements[:,0] # prima colonna

tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])

In [28]:
X_elements[:,-1] # ultima colonna

tensor([ 3.,  6.,  9., 12., 15., 18., 21., 24., 27., 30.])

In [29]:
X_elements[:,0:2] # prima e seconda colonna

tensor([[ 1.,  2.],
        [ 2.,  4.],
        [ 3.,  6.],
        [ 4.,  8.],
        [ 5., 10.],
        [ 6., 12.],
        [ 7., 14.],
        [ 8., 16.],
        [ 9., 18.],
        [10., 20.]])

In [30]:
X_elements[3:5,0:2] # prima e seconda colonna, quarta e quinta riga

tensor([[ 4.,  8.],
        [ 5., 10.]])

è inoltre possibile modificare solo una parte del tensore sfruttando questo trucco

In [31]:
X_elements[3:5,0:1]*=2

In [32]:
X_elements

tensor([[ 1.,  2.,  3.],
        [ 2.,  4.,  6.],
        [ 3.,  6.,  9.],
        [ 8.,  8., 12.],
        [10., 10., 15.],
        [ 6., 12., 18.],
        [ 7., 14., 21.],
        [ 8., 16., 24.],
        [ 9., 18., 27.],
        [10., 20., 30.]])